# RunPod: A1 (`flat_balanced`), balanced clip sampling, against B1

**An ablation, not a model candidate.** B1 with balanced clip sampling (`data.balance.enabled=true`, `balance.py`): the clips that hold rare faced blocks (iron, coal, diamond, lava, plants) or rare events (crafts, collecting iron, coal and diamond) are drawn up to 8 times as often, so that no such item stays below about 5% of the draws; items from fewer than 500 separate scenes are left as they are, which keeps the model from memorising the few transitions there are. The weights come from the simulator's `state`, a label a real agent would not have: the run asks what the latent keeps when rare things are seen often, not for a model we would use. SIGReg stays 0.09. The questions: do the faced rare tiles and the iron and coal collects reach the latent (`probe_report.md`: the six-tile reader, the events, the change test), and does that change the cheap planning on set A. A1 was trained on 2026-10-27, before the ids, into the folder `flat_balanced`, which it keeps.

Each variant and ablation has its own notebook and its own pod (A100, about $1.8/h): two runs train in parallel. Once per pod, in a terminal: `git clone https://github.com/quastAI/AAC.git /root/AAC`, then open this notebook from there. The paths (what lives on the network volume and what on the local disk) and the helpers are in `notebooks/pod.py`, shared with `runpod_B1.ipynb`.

**Before the variants**: `runpod_B1.ipynb` has run B1 (its probe suite and the cheap P1 on task set A, with the random baseline) and uploaded the results. This notebook takes them from the Hub; what it does not find there it runs itself (the same jobs, skipped when done), which costs this pod the time.

| step | what |
|---|---|
| 0 | configuration, Hugging Face login, the PyTorch environment, the dataset from the Hub |
| 1 | B1 and its results on set A from the Hub, a **smoke run** of this run on the real files (a few batches, then the probes) |
| 2 | the run of `experiments/run.py run A1`, detached: training (10 epochs, about 3.3 hours; resumes after a pod restart), the probe suite at its end, the cheap P1 on set A, the comparison with B1 and the random baseline (`reports/A1_A_val.md`, `A1_models.md`) |
| 3 | this run, B1's and the baseline's results and the reports to the Hub |

A step that finished is skipped when you run its cell again.

In [ ]:
# --- 0. Configuration ------------------------------------------------------------------------------------------------
import os, shutil, sys, time
from pathlib import Path

EXPERIMENT = "A1"                          # the id: the experiment and its one trained run in experiments/experiments.yaml (README, "Ids")
REFERENCE = "B1"                            # the flat baseline, the model this run is compared with (a reference run of the experiment)

REPO = Path("/root/AAC")                    # the git clone, on the local disk (pod.py has the other paths and why they are where they are)
sys.path.insert(0, str(REPO / "notebooks"))
import pod

HF_REPO_ID = "quastAI/craftax-classic-pqn-rnn-v1"   # the dataset repo of runpod_dataset.ipynb (data/), runs go to runs/
HF_PRIVATE = True
SEEDS = "42"                                # one seed per variant (settled on 2026-10-20)
MAX_EPOCHS = 10                             # as B1: 155k steps, about 3.3 hours on an A100; the cosine schedule ends here
USE_WANDB = True                            # live curves in Weights & Biases (project hjepa-flat); False = the metrics CSV only

In [ ]:
# --- 0. Paths, Hugging Face login, the PyTorch environment (installed again on a new pod) and the dataset ----------------
pod.configure(repo=REPO, seeds=SEEDS)
from pod import DATA, H_JEPA, PYTHON, RUNNER, launch, progress, run_dir, running, sh, show, wait

TRAIN_OVERRIDES = pod.train_overrides(USE_WANDB)   # Hydra overrides for every training run
api = pod.login(HF_REPO_ID, HF_PRIVATE, USE_WANDB)
pod.install(api, HF_REPO_ID, USE_WANDB)
SPEC = pod.run_spec(EXPERIMENT, EXPERIMENT)        # the run's training config and overrides (its run is named by the id)
FOLDER, REFERENCE_FOLDER = pod.folder_of(EXPERIMENT), pod.folder_of(REFERENCE)   # checkpoint folders: B1 and A1 keep the ones they were trained in
print(EXPERIMENT, FOLDER, SPEC["config"], *SPEC.get("overrides", []))

## 1. B1 from the Hub, a smoke run

B1's folder (`flat`: the model, its reports, its planning results on set A) and the random baseline on set A come from `runs/` of the Hub repo. A probe report of an older suite is moved aside (`health/probe_suite_v1/`), so that B1 is probed again on this pod. The smoke run trains this run for 20 batches with its own overrides and runs the probe suite on 16 clips: a wrong override or a crash in the run's code path shows here, not after hours.

In [ ]:
# --- 1. B1 and its results on set A from the Hub (what this pod's volume has is kept) --------------------------------------------
pod.fetch(api, HF_REPO_ID, f"{REFERENCE_FOLDER}/seed{SEEDS.split(',')[0]}")   # the model, its reports and planning results
pod.fetch(api, HF_REPO_ID, "random/eval_A_val", required=False)        # the random baseline on set A
for model in pod.traced_references(EXPERIMENT):   # trained in another experiment and read at the same steps (B2 for V3): only their race/ files
    pod.fetch(api, HF_REPO_ID, f"{pod.folder_of(model)}/seed{SEEDS.split(',')[0]}", required=False, only="race/")
reference = run_dir(REFERENCE_FOLDER)
planned = (reference / "eval_A_val_craftax_plan" / "planning_report.md").exists()
print("B1 on set A:", "done" if planned else "missing, planned on this pod (runpod_B1.ipynb plans it once for all variants)")
print("B1's probe report:", "current" if pod.current_probe_report(reference) else "missing or older, probed again on this pod")

In [ ]:
# --- 1. Smoke run of this run on the real files (the full output is in LOGS/smoke.log) ----------------------------------
SMOKE = f"{EXPERIMENT}_smoke"
shutil.rmtree(run_dir(SMOKE), ignore_errors=True)
t0 = time.time()
sh([PYTHON, "main_hjepa.py", "--config-name", SPEC["config"], *SPEC.get("overrides", []), f"output_model_name={SMOKE}",
    f"seed={SEEDS.split(',')[0]}", "trainer.max_epochs=1", "+trainer.limit_train_batches=20", "+trainer.limit_val_batches=2",
    "health.probe_clips=16", *TRAIN_OVERRIDES, "wandb.enabled=false"], cwd=H_JEPA, quiet=True, log="smoke")   # no W&B run for a smoke run
out = run_dir(SMOKE)
assert (out / f"{SMOKE}_object.ckpt").exists(), "the smoke run wrote no model"
assert (out / "health" / "report.md").exists() and (out / "health" / "probe_report.md").exists(), "the smoke run wrote no reports"
print(f"smoke run ok in {time.time() - t0:.0f} s: {out}")

## 2. Training, probe suite, cheap P1 against B1

One detached job chain (`run.py run EXPERIMENT`, the id of the configuration cell): the task set A, the training (the probe suite runs at its end), B1 and the random baseline on set A when they are missing, this run on set A, then the two comparison reports. If the pod restarts, run the start cell again: the training resumes from `lightning_resume/last.ckpt`, the finished jobs are skipped. Two epochs in a row with a critical finding stop the training (`health.stop_after_critical_epochs`); the model is still saved and the jobs after it run, so read `report.md` first.

In [ ]:
# --- 2. Start (or resume) the run, detached ----------------------------------------------------------------------------------
OVERRIDES = [f"trainer.max_epochs={MAX_EPOCHS}", *TRAIN_OVERRIDES]
sh([*RUNNER, "plan", EXPERIMENT, "--seeds", SEEDS, "--override", *OVERRIDES])
launch(EXPERIMENT, [*RUNNER, "run", EXPERIMENT, "--mode", "local", "--seeds", SEEDS, "--override", *OVERRIDES])

In [ ]:
# --- 2. Monitor (run as often as you like) -----------------------------------------------------------------------------------------
print(EXPERIMENT, "RUNNING" if running(EXPERIMENT) else "not running")
print("\n".join(progress(EXPERIMENT, "run  ", 1)))   # the job that runs now
print("\n".join(progress(EXPERIMENT, "Epoch", 2) + progress(EXPERIMENT, "[health]", 3) + progress(EXPERIMENT, "[probe suite]", 2) + progress(EXPERIMENT, "success_rate", 2)))
for p in sorted((run_dir(FOLDER) / "health").glob("epoch_*.json"))[-1:]:
    print("last epoch report:", p.name)

In [ ]:
# --- 2. Loss curves (run as often as you like, also while it trains: the metrics CSV is written as it goes) ----------------------
from IPython.display import Image, display
sh([PYTHON, REPO / "analysis" / "plot_training.py", "--run", run_dir(FOLDER)])
display(Image(filename=str(run_dir(FOLDER) / "health" / "loss_curves.png")))

In [ ]:
# --- 2. Wait for the chain, then the reports: training, probe suite, the comparisons with B1 ------------------------------------------
wait(EXPERIMENT, "Epoch")
show(run_dir(FOLDER) / "health" / "report.md", 40)
show(run_dir(FOLDER) / "health" / "probe_report.md", 120)
for report in sorted((DATA / "reports").glob(f"{EXPERIMENT}_*.md")):
    print("\n=====", report.name)
    show(report, 80)

## 3. Upload

This run, B1's folder (planning results or a probe report made on this pod) and the random baseline go to `runs/`, this run's reports to `reports/` of the Hub repo. Left out: the resumable training state, the per-epoch models and the smoke run.

In [ ]:
# --- 3. Runs and reports to the Hub --------------------------------------------------------------------------------------------------------
pod.upload(api, HF_REPO_ID, folders=[FOLDER, REFERENCE_FOLDER, "random"], reports=[f"{EXPERIMENT}_*.md"])